# Connecting your SUT

This notebook walks through the three integration levels for connecting a System Under Test (SUT) to the GMSH harness. The same material is covered in §3.5 of `knowlytix/harness/testing/USER_GUIDE.md` and in `gmsh_testing_ext/chapters/ch11.tex` (SUT Abstraction). Here we run it end-to-end against a tiny stub agent so the contract is concrete.

By the end you will have:

1. Built a `TargetProfile` that identifies your SUT.
2. Wrapped a stub agent in an `AgentAdapter` subclass that emits a standard `SUTTrace`.
3. Demonstrated all three integration levels (passive observation, active interception, full simulation) against the same stub.
4. Confirmed via assertion that the gateway captured at least one decision.

## 1. Preamble

In [ ]:
from __future__ import annotations

import sys
import time
from pathlib import Path

# Make sure the kg-memory project root is importable.
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name and not (PROJECT_ROOT / "harness" / "doe" / "__init__.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

device = "cpu"  # SUT-connection story does not need GPU
print(f"project root: {PROJECT_ROOT}")
print(f"device:       {device}")

In [ ]:
from knowlytix.harness.testing import (
    AgentAdapter, SUTTrace, ToolCallRecord,
    TargetProfile, IsolationRequirements,
    ToolGateway, FaultProfile,
    EnvironmentSimulator, ToolEndpointSpec, StateTransition,
)
print("imports OK")

## 2. Concept

The harness only sees a SUT through one method:

```python
def evaluate(self, question: str, context: str) -> SUTTrace: ...
```

Whatever the underlying system is — a bare LLM, a tool-using agent, a RAG pipeline, a browser-driven UI — its adapter must accept a question and a context string and return a populated `SUTTrace`. The trace carries the answer (`raw_output`), the tool sequence (`tool_calls`), and timing metadata (`latency_ms`, `token_count`).

There are three escalation levels for what the harness can verify:

- **Level 1 (passive observation).** The adapter emits a `SUTTrace`, the harness reads the answer and the observed tool sequence. No interception, no fault injection. This is enough to verify final answers and tool ordering.

- **Level 2 (active interception).** A `ToolGateway` is installed so every tool call passes through gateway hooks first. The gateway can block calls (policy), inject faults (latency, errors, mock responses), and write a full audit transcript. This is what you need to verify recovery paths and policy compliance.

- **Level 3 (full simulation).** The real tool implementations are replaced by an `EnvironmentSimulator` with declarative `ToolEndpointSpec` entries. Every tool call hits a deterministic synthetic endpoint. This is what audit-grade replay demands, but it costs you the work of reimplementing each tool's behavior.

Start at Level 1. Move up only when the next level unlocks a verification you actually need. The remainder of the notebook demonstrates each level against the same stub agent.

## 3. Code §A — Build a `TargetProfile`

The target profile is the SUT's audit identity: name, version, type, isolation requirements. It gets recorded in audit packets and replay manifests so reruns are reproducible.

In [ ]:
target = TargetProfile(
    name="StubAgent",
    version="0.1.0",
    target_type="agent",
    endpoint="local",
    model_name="stub-model-v1",
    capabilities=["tool_use"],
    timeout_ms=30_000,
    max_concurrent=1,
    isolation=IsolationRequirements(
        network_isolated=True,
        sandbox_filesystem=True,
        synthetic_data_only=True,
    ),
    metadata={"notebook": "connecting_a_sut"},
)

print(target.summary())
print("target_id:", target.target_id)
print("isolation:", target.isolation.to_dict())

## 4. Code §B — Define a stub agent and an `AgentAdapter` subclass

The stub agent below pretends to call a `data_lookup` tool, then composes a one-line answer. The adapter subclass is responsible for translating the agent's internal log into the standard `ToolCallRecord` shape and packaging the result as a `SUTTrace`.

In [ ]:
class StubAgent:
    """A 20-line stand-in for a real tool-using agent."""

    def __init__(self):
        self._records: list[dict] = []

    def ask(self, question: str, context: str) -> str:
        """Call data_lookup, then answer."""
        t0 = time.time()
        # Pretend to call data_lookup
        result = '{"auc": 0.8412, "risk": "High"}'
        self._records.append({
            "tool_name": "data_lookup",
            "arguments": {"entity": "MDL-001"},
            "result": result,
            "latency_ms": (time.time() - t0) * 1000,
        })
        return "0.8412"

    @property
    def tool_calls(self) -> list[dict]:
        return list(self._records)

    def reset(self) -> None:
        self._records = []


class StubAgentAdapter(AgentAdapter):
    """Translate StubAgent records into a SUTTrace."""

    def __init__(self, agent: StubAgent):
        self.agent = agent
        self.harness = None     # AgentAdapter base expects this attribute
        self.gateway = None

    def evaluate(self, question: str, context: str) -> SUTTrace:
        self.agent.reset()
        t0 = time.time()
        try:
            answer = self.agent.ask(question, context)
            return SUTTrace(
                raw_output=answer,
                tool_calls=[ToolCallRecord(**rec)
                            for rec in self.agent.tool_calls],
                latency_ms=(time.time() - t0) * 1000,
            )
        except Exception as exc:
            return SUTTrace(
                raw_output="",
                latency_ms=(time.time() - t0) * 1000,
                error=str(exc),
            )


agent = StubAgent()
adapter = StubAgentAdapter(agent)
trace = adapter.evaluate(
    "What is the test AUC of MDL-001?",
    "MDL-001 has test_auc=0.8412 and risk=High",
)
print("answer:    ", trace.raw_output)
print("latency_ms:", round(trace.latency_ms, 3))
print("tool_calls:")
for tc in trace.tool_calls:
    print(" ", tc.tool_name, tc.arguments, "->", tc.result)

## 5. Code §C — Three integration levels

We run the same stub agent at each level and record what we learn from the trace.

### Level 1: Passive observation

Call `adapter.evaluate(...)` and read the trace. No gateway, no simulator. The harness can verify the answer string and the observed tool sequence.

In [ ]:
level1_trace = adapter.evaluate(
    "What is the test AUC of MDL-001?",
    "MDL-001 has test_auc=0.8412",
)
print("Level 1 inspectable:")
print("  raw_output:", level1_trace.raw_output)
print("  n tool_calls:", len(level1_trace.tool_calls))
print("  latency_ms: ", round(level1_trace.latency_ms, 3))

### Level 2: Active interception with `ToolGateway`

We install a gateway in front of a `MockHarness`-backed adapter so we can demonstrate a real intercept. The gateway records an allow/block/inject decision per tool call. We use `MockHarness` here because it implements the `harness.hooks` interface that `ToolGateway.install` expects.

In [ ]:
from knowlytix.harness.fixtures import MockHarness
from knowlytix.knowledge.store import GMSExpertStore

# A small store is enough — the gateway only uses it for plausibility scoring.
store = GMSExpertStore(device="cpu")
store.learn_triples([
    ("agent", "has_capability", "data_lookup"),
    ("agent", "has_capability", "risk_check"),
])

mock_harness = MockHarness(recovery_mode="fallback")

fault = FaultProfile(
    tool_pattern="risk_check",
    error_rate=1.0,
    error_message="Simulated outage",
)
gateway = ToolGateway(
    store=store,
    fault_profiles=[fault],
    run_id="nb_run_001",
)
level2_adapter = AgentAdapter(mock_harness, gateway=gateway)

level2_trace = level2_adapter.evaluate(
    "What is the risk rating of MDL-001?",
    "MDL-001 CreditScore-XGB has risk=High, AUC=0.8412",
)
transcript = gateway.get_transcript()

print("Level 2 inspectable + intercept-able:")
print("  raw_output:    ", level2_trace.raw_output[:80])
print("  n tool_calls:  ", len(level2_trace.tool_calls))
print("  n decisions:   ", len(transcript.decisions))
for d in transcript.decisions:
    print(f"    {d.tool_name}: {d.action} -- {d.reason}")

### Level 3: Full simulation with `EnvironmentSimulator`

We define two synthetic endpoints and route the agent's tool calls through them. The simulator returns deterministic responses; state changes are observable for side-effect oracles.

In [ ]:
env = EnvironmentSimulator(
    tool_specs=[
        ToolEndpointSpec(
            tool_name="data_lookup",
            initial_state={"loaded": 0},
            default_response='{"status": "not_found"}',
            state_transitions=[
                StateTransition(
                    trigger_action="data_lookup",
                    trigger_condition="args.entity == 'MDL-001'",
                    state_mutation={"loaded": 1, "last": "MDL-001"},
                    response_template='{"auc": 0.8412, "risk": "High"}',
                ),
            ],
        ),
        ToolEndpointSpec(
            tool_name="risk_check",
            default_response='{"risk_level": "unknown"}',
        ),
    ],
)
env.reset(seed=42)

before_state = env.get_state()
resp_a = env.handle_tool_call("data_lookup", {"entity": "MDL-001"})
resp_b = env.handle_tool_call("risk_check", {"model_id": "MDL-001"})
after_state = env.get_state()
diff = env.get_state_diff(before_state, after_state)

print("Level 3 deterministic responses:")
print("  data_lookup ->", resp_a)
print("  risk_check  ->", resp_b)
print("State diff:")
print("  added:  ", diff["added"])
print("  changed:", diff["changed"])

## 6. Visualization — what each level reveals

A side-by-side bar chart of the three capabilities each level unlocks: tool calls inspectable, tool calls fault-injectable, tool calls deterministic.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

levels = ["Level 1\nObservation", "Level 2\nInterception", "Level 3\nSimulation"]
inspectable = [1, 1, 1]
fault_injectable = [0, 1, 1]
deterministic = [0, 0, 1]

x = np.arange(len(levels))
w = 0.25

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(x - w, inspectable, w, label="inspectable")
ax.bar(x, fault_injectable, w, label="fault-injectable")
ax.bar(x + w, deterministic, w, label="deterministic")
ax.set_xticks(x)
ax.set_xticklabels(levels)
ax.set_ylim(0, 1.2)
ax.set_ylabel("capability (1 = unlocked)")
ax.set_title("What each integration level reveals")
ax.legend(loc="upper left")
fig.tight_layout()
plt.show()

## 7. Final assertion

After Level 2 the gateway transcript must contain at least one decision. If it does not, the gateway never saw a tool call, which means the adapter wiring is wrong.

In [ ]:
assert len(transcript.decisions) >= 1, (
    "Gateway captured no decisions -- adapter is not wired correctly"
)
print(f"OK: gateway recorded {len(transcript.decisions)} decision(s)")

## 8. Where to go next

- §4 (Tutorial 1) — your first benchmark run with this adapter wired in.
- §5 (Tutorial 2) — fault profile semantics, YAML loading, full gateway transcript.
- §12 (Tutorial 9) — extended `TargetProfile` walkthrough, including config integration.
- §13 (Tutorial 10) — full `EnvironmentSimulator` API: state transitions, scheduled events, fixtures, clock control.
- `gmsh_testing_ext/chapters/ch11.tex` — conceptual deep-dive on SUT abstraction, adapters, traces.